In [ ]:
import os
import time
import mne
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import random
import optuna

# ====================================================================
# 1. CONFIGURAÇÕES E PARÂMETROS
# ====================================================================
DATA_PATH = 'data/physionet_data/'
TARGET_RUNS = ['04', '08', '12']
TARGET_CHANNELS = ['C3..', 'C4..', 'Cz..']
L_FREQ, H_FREQ = 8, 30
FS = 160

# --- Parâmetros da Otimização ---
NUM_SUBJECTS_FOR_TUNING = 80 #quantas pastas você deseja utilizar (S001 até S109)
N_TRIALS = 100 #Número total de trials
EPOCHS = 60 #Número de épocas por trial
BATCH_SIZE = 16 #BatchSize por trial -> recomendo testar com 16 e 32

# ====================================================================
# 2. FUNÇÕES AUXILIARES
# ====================================================================

#Criação do modelo variando as camadas convolucionais, unidades LSTM e taxa de dropout (OPTUNA FAZ ISSO, mas é possível trocar a estrutura do modelo)
def create_optimized_model(input_shape, num_classes, conv_layers_count, lstm_units, dropout_rate):
    """Cria um modelo flexível com base nos hiperparâmetros sugeridos."""
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    for i in range(conv_layers_count):
        filters = 32 * (2**i)
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        
        # Adiciona o Max Pooling apenas se as dimensões espaciais (altura e largura) 
        # forem maiores que 1. Isso evita o erro quando uma dessas dimensões se 
        # torna 1 após sucessivas operações de pooling.
        if x.shape[1] > 1 and x.shape[2] > 1:
            x = layers.MaxPooling2D((2, 2))(x)
            
        x = layers.BatchNormalization()(x)
    
    last_conv_shape = x.shape
    # A linha abaixo achata as dimensões de altura e largura para a camada LSTM
    # Ex: (None, 8, 10, 64) -> (None, 80, 64)
    reshaped_shape = (last_conv_shape[1] * last_conv_shape[2], last_conv_shape[3])
    x = layers.Reshape(reshaped_shape)(x)
    
    x = layers.Bidirectional(layers.LSTM(lstm_units))(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def generate_spectrogram_for_window(window_data, frame_length, frame_step, fft_length, fs=FS):
    """Gera um único espectrograma para uma janela de dados já extraída."""
    num_channels = window_data.shape[0]
    channel_spectrograms = []
    for i in range(num_channels):
        stft = np.abs(tf.signal.stft(
            window_data[i], frame_length=frame_length, frame_step=frame_step, fft_length=fft_length))
        channel_spectrograms.append(stft)
    return np.stack(channel_spectrograms, axis=-1)

# ====================================================================
# 3. FUNÇÃO OBJECTIVE PARA O OPTUNA
# ====================================================================
def objective(trial):
    """Função que o Optuna irá otimizar."""
    print(f"\n---> INICIANDO ENSAIO (TRIAL) #{trial.number} <---")
    

    janela_s = trial.suggest_float('janela_s', 1.5, 2.0) #Janela Maior onde a janela deslizante irá atuar
    
    # Janela da STFT (50ms a 200ms)
    frame_length_ms = trial.suggest_int('frame_length_ms', 50, 200, step=10) #janela deslizante onde será aplicada a stft
    
    # Overlap da janela da STFT (25% a 75%)
    stft_overlap_ratio = trial.suggest_float('stft_overlap_ratio', 0.25, 0.75) #overlap (0.25 a 0.75 são valores interessantes)

    # Outros parâmetros
    fft_length = trial.suggest_categorical('fft_length', [64, 128, 256]) # Tamanho do FFT (64, 128 ou 256) -> sugestão do gemini
    conv_layers_count = trial.suggest_int('conv_layers', 1, 6) #Aqui talvez seja interessante aumentar o número de camadas convolucionais mínimas para 4 e aumentar o máximo para um valor maior que 6
    lstm_units = trial.suggest_int('lstm_units', 32, 128, step=32) # Unidades LSTM (32, 64, 96, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7) # Taxa de dropout (0.2 a 0.7)

    # Conversão dos parâmetros para o formato de amostras, igual é dito no docs que enviei sobre frame_step ser em amostras (as bibiliotecas de STFT utilizam amostras)
    window_size = int(janela_s * FS)
    frame_length = int((frame_length_ms / 1000) * FS)
    frame_step = int(frame_length * (1 - stft_overlap_ratio))
    frame_step = max(1, frame_step)

    #Aqui é feita uma checagem para garantir que o frame_length não seja maior que o tamanho da janela (não perder tempo com dados inválidos)
    if frame_length > fft_length or frame_length <= 0:
        raise optuna.exceptions.TrialPruned("Parâmetros de STFT inválidos.")
    
    print(f"Parâmetros: janela_s={janela_s:.2f}s, frame_len={frame_length_ms}ms, frame_step={frame_step}, overlap={stft_overlap_ratio:.2f}, ...")

    #Coleta e Processamento com lógica de centralização e ajuste de tamanho
    #encurta o sinal para as frequências de interesse
    all_spectrograms, all_labels = [], []
    freqs = np.fft.rfftfreq(n=fft_length, d=1/FS)
    freq_indices = np.where((freqs >= L_FREQ) & (freqs <= H_FREQ))[0]

    #vai iterar sobre os sujeitos e runs especificados
    for subject_num in range(1, NUM_SUBJECTS_FOR_TUNING + 1):
        subject_id = f'S{subject_num:03d}'
        subject_path = os.path.join(DATA_PATH, subject_id)
        if not os.path.isdir(subject_path): continue
        for run_num in TARGET_RUNS:
            file_path = os.path.join(subject_path, f'{subject_id}R{run_num}.edf')
            if os.path.exists(file_path):
                try:
                    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
                    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
                    
                    for ann in raw.annotations:
                        label = ann['description']
                        if label in ['T0', 'T1', 'T2']:
                            event_center_s = ann['onset'] + (ann['duration'] / 2)
                            window_start_s = event_center_s - (janela_s / 2)
                            window_end_s = event_center_s + (janela_s / 2)

                            if window_start_s >= 0 and window_end_s <= raw.times[-1]:
                                start_sample, stop_sample = raw.time_as_index([window_start_s, window_end_s])
                                
                                window_data = raw.get_data(picks=TARGET_CHANNELS, start=start_sample, stop=stop_sample)
                                
                                # Bloco de ajuste para garantir o tamanho exato da janela
                                current_len = window_data.shape[1]
                                if current_len > window_size:
                                    window_data = window_data[:, :window_size]
                                elif current_len < window_size:
                                    padding_needed = window_size - current_len
                                    window_data = np.pad(window_data, ((0, 0), (0, padding_needed)), 'constant')

                                spectrogram = generate_spectrogram_for_window(window_data, frame_length, frame_step, fft_length)
                                
                                if spectrogram.shape[1] > 0:
                                    spectrogram_cropped = spectrogram[freq_indices, :, :]
                                    all_spectrograms.append(spectrogram_cropped)
                                    all_labels.append(label)
                except Exception:
                    continue
    
    # --- Balanceamento, Preparação, Treinamento e Avaliação ---
    if len(all_spectrograms) < 50:
        raise optuna.exceptions.TrialPruned("Não foram gerados dados suficientes com estes parâmetros.")

    X = np.array(all_spectrograms)
    y_str = np.array(all_labels)

    unique_labels, counts = np.unique(y_str, return_counts=True)
    if len(unique_labels) < 3: raise optuna.exceptions.TrialPruned("Amostras de todas as 3 classes não foram encontradas.")
    
    labels_t1_t2 = (y_str == 'T1') | (y_str == 'T2')
    indices_t1_t2, indices_t0 = np.where(labels_t1_t2)[0], np.where(y_str == 'T0')[0] 
    
    #balanceamento dos dados (T0 tem o dobro de amostras que T1 e T2, então é preciso balancear)
    target_count = len(indices_t1_t2) // 2
    if len(indices_t0) > target_count and target_count > 0:
        indices_t0_balanced = np.random.choice(indices_t0, size=target_count, replace=False)
        final_indices = np.concatenate([indices_t0_balanced, indices_t1_t2])
    else:
        final_indices = np.arange(len(y_str))
        
    X_balanced, y_str_balanced = X[final_indices], y_str[final_indices]
    
    encoder = LabelEncoder()
    y_balanced = encoder.fit_transform(y_str_balanced)
    
    permutation = np.random.permutation(len(X_balanced))
    X_balanced, y_balanced = X_balanced[permutation], y_balanced[permutation]

    X_train, X_val, y_train, y_val = train_test_split(X_balanced, y_balanced, test_size=0.25, random_state=42, stratify=y_balanced)

    if X_train.shape[0] == 0 or X_train.shape[1] <= 0 or X_train.shape[2] <= 0:
        raise optuna.exceptions.TrialPruned("Dados de treino com shape inválido após processamento.")
        
    model = create_optimized_model(X_train.shape[1:], len(encoder.classes_), conv_layers_count, lstm_units, dropout_rate)
    
    model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)]) #EarlyStopping para evitar iterações desnecessárias
    
    y_pred_probs = model.predict(X_val)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    
    #Aqui é possível adicionar mais métricas 
    f1 = f1_score(y_val, y_pred_classes, average='weighted')
    print(f"Ensaio #{trial.number} concluído. F1-Score: {f1:.4f}")
    
    return f1

# ====================================================================
# 4. EXECUÇÃO DO ESTUDO DE OTIMIZAÇÃO
# ====================================================================
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS)

# --- Exibição dos Resultados ---
print("\nOtimização Concluída!")
print(f"Melhor F1-Score: {study.best_value:.4f}")
print("Melhores Hiperparâmetros encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

c:\Users\luize\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-10 16:51:51,232] A new study created in memory with name: no-name-116edb8f-ae2a-49e2-952f-94ab18e1f344



---> INICIANDO ENSAIO (TRIAL) #0 <---
Parâmetros: janela_s=2.90s, frame_len=140ms, frame_step=14, overlap=0.35, ...
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


[I 2025-07-10 16:53:09,338] Trial 0 finished with value: 0.16666666666666666 and parameters: {'janela_s': 2.8991283558782315, 'frame_length_ms': 140, 'stft_overlap_ratio': 0.35215626714803827, 'fft_length': 64, 'conv_layers': 2, 'lstm_units': 32, 'dropout_rate': 0.6202521442476077}. Best is trial 0 with value: 0.16666666666666666.


Ensaio #0 concluído. F1-Score: 0.1667

---> INICIANDO ENSAIO (TRIAL) #1 <---
Parâmetros: janela_s=2.27s, frame_len=50ms, frame_step=5, overlap=0.36, ...
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


[I 2025-07-10 16:54:45,143] Trial 1 finished with value: 0.16666666666666666 and parameters: {'janela_s': 2.2677033944913134, 'frame_length_ms': 50, 'stft_overlap_ratio': 0.35738339552878795, 'fft_length': 128, 'conv_layers': 3, 'lstm_units': 64, 'dropout_rate': 0.34855153007818807}. Best is trial 0 with value: 0.16666666666666666.


...

Ensaio #46 concluído. F1-Score: 0.3297

---> INICIANDO ENSAIO (TRIAL) #47 <---
Parâmetros: janela_s=1.65s, frame_len=130ms, frame_step=8, overlap=0.58, ...
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


[I 2025-07-10 18:31:37,241] Trial 47 finished with value: 0.16666666666666666 and parameters: {'janela_s': 1.6462417147543933, 'frame_length_ms': 130, 'stft_overlap_ratio': 0.5769410355751541, 'fft_length': 64, 'conv_layers': 6, 'lstm_units': 32, 'dropout_rate': 0.5562845059076841}. Best is trial 46 with value: 0.32966630714869694.


Ensaio #47 concluído. F1-Score: 0.1667

---> INICIANDO ENSAIO (TRIAL) #48 <---
Parâmetros: janela_s=1.90s, frame_len=110ms, frame_step=8, overlap=0.52, ...


[W 2025-07-10 18:31:40,689] Trial 48 failed with parameters: {'janela_s': 1.9039959560382098, 'frame_length_ms': 110, 'stft_overlap_ratio': 0.5231019338546724, 'fft_length': 256, 'conv_layers': 6, 'lstm_units': 64, 'dropout_rate': 0.4636715091116646} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\luize\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\luize\AppData\Local\Temp\ipykernel_18572\3431956712.py", line 126, in objective
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
  File "c:\Users\luize\AppData\Local\Programs\Python\Python310\lib\site-packages\mne\io\base.py", line 1174, in filter
    return super().filter(
  File "<decorator-gen-56>", line 10, in filter
  File "c:\Users\luize\AppData\Local\Programs\Python\Python310\lib\site-packages\mne\filter.py", line 2562, in filter
    filter_data(
  File "

KeyboardInterrupt: 